In [19]:
from langgraph.graph import StateGraph,START,END
from langchain_groq import ChatGroq
from typing import TypedDict,Annotated
from pydantic import BaseModel , Field
from dotenv import load_dotenv
import operator

In [20]:
load_dotenv()

model = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

In [21]:
class EvaluationSchema(BaseModel):

  feedback:str = Field(description="Detailed feedback for the essay")
  score:float = Field(description="Score out of 10", ge=0, le=10)
  

In [22]:
structured_model = model.with_structured_output(EvaluationSchema)

In [23]:
essay = """
Artificial Intelligence (AI) is one of the most important technologies of the 21st century. It enables machines and computer systems to perform tasks that normally require human intelligence, such as learning, problem-solving, decision-making, and understanding language. India is emerging as an important player in the global AI revolution because of its strong IT industry, talented workforce, growing startup ecosystem, and increasing government support.
India is contributing to AI development through research, innovation, education, and practical applications. Indian engineers, researchers, and entrepreneurs are developing AI-powered solutions in healthcare, agriculture, education, banking, transportation, and cybersecurity.

India's large pool of software developers and technology professionals also helps global companies develop and deploy AI systems. Indian technology companies are using AI to automate business processes, improve customer service, analyze data, and increase productivity.

The Indian government is taking several initiatives to promote AI. The IndiaAI Mission, approved in 2024, aims to strengthen the country's AI ecosystem by supporting computing infrastructure, innovation, skills development, and access to AI resources. The government also promotes digital transformation through initiatives such as Digital India.

These efforts can help startups, researchers, students, and businesses build AI solutions suited to India's needs. They also encourage responsible AI development, data protection, and wider access to technology.
"""


In [ ]:
prompt = f"""
Evaluate the language quality of the following essay.
Provide detailed feedback and assign a score out of 10.

Essay:
{essay}
"""

result = structured_model.invoke(prompt)


The essay presents a clear and coherent overview of India’s role in the AI landscape, but the language quality shows room for improvement. Strengths: • The structure is logical, moving from a general introduction of AI to India’s contributions, industry involvement, and government initiatives. • Vocabulary is appropriate for the topic, with terms such as "AI‑powered solutions," "ecosystem," and "digital transformation" used correctly. • Sentences are generally well‑formed and the ideas are easy to follow.

Weaknesses: • Repetition of the phrase "AI" and similar constructions (e.g., "AI development," "AI solutions") makes the prose feel redundant. Varying synonyms or restructuring sentences would improve flow. • Some sentences are overly simple or lack variety in length and complexity, which reduces stylistic richness (e.g., "India is emerging as an important player…" and "Indian engineers, researchers, and entrepreneurs are developing…"). Adding subordinate clauses or transitional phra

In [31]:
class UPSCState(TypedDict):

  essay:str
  language_feedback:str
  analysis_feedback: str
  clarity_feedback:str
  overall_feedback:str
  individual_scores:Annotated[list[int],operator.add]
  avg_score:float

In [41]:
def evaluate_language(state:UPSCState):
  prompt = f"""
Evaluate the language quality of the following essay.
Provide detailed feedback and assign a score out of 10.

Essay:
{state['essay']}
"""
  output = structured_model.invoke(prompt)

  return {'language_feedback':output.feedback,'individual_scores':[output.score]}

In [42]:
def evaluate_analysis(state:UPSCState):
  prompt = f"""
Evaluate the depth of analysis of the following essay.
Provide detailed feedback and assign a score out of 10.

Essay:
{state['essay']}
"""
  output = structured_model.invoke(prompt)

  return {'analysis_feedback':output.feedback,'individual_scores':[output.score]}

In [43]:
def evaluate_thought(state:UPSCState):
  prompt = f"""
Evaluate the clarity of thought of the following essay.
Provide detailed feedback and assign a score out of 10.

Essay:
{state['essay']}
"""
  output = structured_model.invoke(prompt)

  return {'clarity_feedback':output.feedback,'individual_scores':[output.score]}

In [44]:
def final_evaluation(state:UPSCState):

  # Summary feedback
  prompt = f"Based on the following create a summarized feedback \n language feedback - {state["language_feedback"]} \n depth analysis feedback - {state['analysis_feedback']} \n clarity of thought feedback - {state['clarity_feedback']} " 

  overall_feedback = model.invoke(prompt).content

  avg_score = sum(state['individual_scores'])/len(state['individual_scores'])

  return {'overall_feedback':overall_feedback,'avg_score':avg_score}

In [45]:
graph = StateGraph(UPSCState)

# Add Nodes
graph.add_node('evaluate_language',evaluate_language)
graph.add_node('evaluate_analysis',evaluate_analysis)
graph.add_node('evaluate_thought',evaluate_thought)
graph.add_node('final_evaluation',final_evaluation)

# Create edges 
graph.add_edge(START, 'evaluate_language')
graph.add_edge(START, 'evaluate_analysis')
graph.add_edge(START, 'evaluate_thought')

graph.add_edge('evaluate_language','final_evaluation')
graph.add_edge('evaluate_analysis','final_evaluation')
graph.add_edge('evaluate_thought','final_evaluation')
graph.add_edge('final_evaluation',END)


workflow = graph.compile()



In [46]:
initial_state = {'essay':essay}

workflow.invoke(initial_state)

{'essay': "\nArtificial Intelligence (AI) is one of the most important technologies of the 21st century. It enables machines and computer systems to perform tasks that normally require human intelligence, such as learning, problem-solving, decision-making, and understanding language. India is emerging as an important player in the global AI revolution because of its strong IT industry, talented workforce, growing startup ecosystem, and increasing government support.\nIndia is contributing to AI development through research, innovation, education, and practical applications. Indian engineers, researchers, and entrepreneurs are developing AI-powered solutions in healthcare, agriculture, education, banking, transportation, and cybersecurity.\n\nIndia's large pool of software developers and technology professionals also helps global companies develop and deploy AI systems. Indian technology companies are using AI to automate business processes, improve customer service, analyze data, and i